# 흐린 위성사진을 선명하게 만들기 (EDSR ×3)

같은 장소를 두 위성이 찍었다. 하나는 흐리고, 하나는 선명하다.

| | 위성 | 한 픽셀이 담는 실제 크기 |
|---|---|---|
| 흐린 쪽 | Sentinel-2 | 10 m |
| 선명한 쪽 | IKONOS | 3.33 m |

10 m짜리를 3.33 m짜리처럼 보이게 만드는 게 목표다. 가로세로 3배씩 키우는 셈이라
**×3 초해상화**라고 부른다. 그냥 확대하면 뿌옇게 뭉개지니까, 선명한 사진을 정답으로 주고
"이런 식으로 채워라"를 AI에게 가르친다.

## 연습문제와 실전시험

여기가 이 실습에서 제일 중요한 부분이다. **가르칠 때와 시험볼 때 문제가 다르다.**

- **연습문제(학습용)** — 선명한 사진을 일부러 3배 줄여서 흐리게 만든 것. 정답에서 만들어낸
  거라 색과 밝기가 정확히 맞는다. 깔끔한 문제집이다.
- **실전시험(검증용)** — 진짜 Sentinel-2 위성이 찍은 흐린 사진. 찍은 날짜도 위성도 다르니까
  색도 다르고 그림자 위치도 다르다.

깔끔한 문제집으로 공부한 학생이 실전에서 점수가 잘 안 오르는 것과 같다. 뒤에서 실제로
그런 일이 벌어지는 걸 보게 된다. **버그가 아니라 이 분야의 진짜 어려움이다.**

## 준비물

| | 개수 | 크기 |
|---|---|---|
| 연습문제 | 40쌍 | 흐린 128픽셀 → 선명한 384픽셀 |
| 실전시험 | 10쌍 | 배울 때 안 본 지역에서만 |
| 최종 테스트 | 1장 | 인천 위성사진. 정답이 없어서 눈으로 본다 |

전부 합쳐 42 MB, 학습은 몇 분이면 끝난다.

> **먼저 GPU를 켜세요**: 위 메뉴에서 런타임 → 런타임 유형 변경 → **T4 GPU** → 저장

## 0. GPU가 켜져 있나 확인

GPU 없이도 돌아가긴 하지만 몇십 배 느리다. 아래를 실행해서 `True`가 나와야 한다.

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU  ', torch.cuda.get_device_name(0))
else:
    raise SystemExit('GPU가 꺼져 있습니다. 런타임 → 런타임 유형 변경 → T4 GPU')

## 1. 실습 자료 가져오기

Colab은 구글이 잠깐 빌려주는 컴퓨터다. **내 컴퓨터의 파일은 안 보인다.** 그래서 매번
자료를 인터넷에서 받아와야 한다.

기본값(`'github'`)이면 아무것도 안 고쳐도 된다. 그냥 실행하면 된다.

In [ ]:
import os, shutil, subprocess, glob

SOURCE      = 'github'                                        # github | drive | url
GITHUB_REPO = 'https://github.com/BWMIN-Hub/SR_practice.git'  # SOURCE='github' 일 때
DRIVE_ZIP   = '/content/drive/MyDrive/sr_colab/colab.zip'     # SOURCE='drive' 일 때
BUNDLE_URL  = ''                                              # SOURCE='url' 일 때


def _mount_drive():
    if not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')


def _find_root(base):
    """dataset/ models/ notebooks/ 를 모두 가진 폴더를 찾는다.

    저장소 루트가 colab/ 자체든, colab/ 을 품은 상위 폴더든 양쪽 다 찾아낸다.
    """
    for d, subs, _ in os.walk(base):
        if {'dataset', 'models', 'notebooks'} <= set(subs):
            return d
    return None


def fetch():
    """번들을 /content 아래로 가져오고 그 루트 경로를 돌려준다."""
    hit = _find_root('/content')
    if hit:                                      # 이미 있으면 다시 받지 않는다
        return hit

    if SOURCE == 'github':
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, '/content/repo'],
                       check=True)
    elif SOURCE in ('drive', 'url'):
        zp = '/content/colab.zip'
        if SOURCE == 'drive':
            _mount_drive()
            assert os.path.exists(DRIVE_ZIP), f'Drive에 없습니다: {DRIVE_ZIP}'
            shutil.copy(DRIVE_ZIP, zp)           # 순차 읽기 1회
        else:
            subprocess.run(['wget', '-q', '-O', zp, BUNDLE_URL], check=True)
        subprocess.run(['unzip', '-q', '-o', zp, '-d', '/content/repo'], check=True)
    else:
        raise ValueError(SOURCE)

    hit = _find_root('/content/repo')
    assert hit, '받은 내용에서 dataset/·models/·notebooks/ 를 가진 폴더를 찾지 못했습니다'
    return hit


ROOT  = fetch()
DATA  = f'{ROOT}/dataset'
MODEL = f'{ROOT}/models/01_edsr_x3'
CODE  = f'{MODEL}/code'
assert os.path.isdir(CODE), CODE
print('번들 준비 완료:', ROOT)

### 결과물은 구글 드라이브에 저장하기

Colab은 **12시간이 지나거나 90분쯤 가만히 두면 연결이 끊긴다.** 그러면 그 안에 있던
파일이 전부 사라진다. 학습에 30분 썼는데 결과가 날아가면 아깝다.

그래서 학습 결과가 저장되는 폴더만 구글 드라이브에 연결해둔다. 파일이 6 MB밖에 안 돼서
용량 걱정은 없다.

드라이브를 안 쓰고 싶으면 아래 `SAVE_TO_DRIVE`를 `False`로 바꾸면 된다.

In [ ]:
SAVE_TO_DRIVE = True   # Drive를 안 쓸 거면 False

if SAVE_TO_DRIVE:
    _mount_drive()
    dst = '/content/drive/MyDrive/sr_colab/experiment'
    os.makedirs(dst, exist_ok=True)
    if not os.path.islink(f'{CODE}/experiment'):
        shutil.rmtree(f'{CODE}/experiment', ignore_errors=True)
        os.symlink(dst, f'{CODE}/experiment')
    print('experiment ->', os.path.realpath(f'{CODE}/experiment'))
else:
    os.makedirs(f'{CODE}/experiment', exist_ok=True)

### 필요한 프로그램 설치

대부분 Colab에 이미 깔려 있다. 위성사진(GeoTIFF) 파일을 읽는 `rasterio` 하나만 더 받으면
된다. 한글이 그래프에서 네모로 깨지지 않도록 폰트도 같이 설치한다.

In [ ]:
!pip install -q rasterio
!apt-get install -qq -y fonts-nanum > /dev/null 2>&1

import matplotlib as mpl
import matplotlib.font_manager as fm

# Colab 기본 이미지에는 한글 폰트가 없어 그래프 제목이 네모로 깨진다
_font = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(_font):
    fm.fontManager.addfont(_font)
    mpl.rc('font', family='NanumGothic')
mpl.rc('axes', unicode_minus=False)

import rasterio
print('rasterio', rasterio.__version__, '| 한글폰트', os.path.exists(_font))

## 2. 자료 살펴보기

무엇을 가지고 시작하는지 먼저 눈으로 본다.

In [ ]:
from collections import Counter

for split in ['training', 'validation']:
    hr = sorted(glob.glob(f'{DATA}/{split}/HR/*.png'))
    lr = sorted(glob.glob(f'{DATA}/{split}/LR_bicubic/X3/*.png'))
    city = Counter(os.path.basename(f).rsplit('_y', 1)[0].rsplit('_', 1)[0] for f in hr)
    print(f'{split:11s} HR {len(hr):3d}장 / LR {len(lr):3d}장   {dict(city)}')

print('\ntest      ', [os.path.basename(f) for f in glob.glob(f'{DATA}/test/*.tif')])

In [ ]:
import imageio.v2 as imageio
import matplotlib.pyplot as plt

def preview(split, n=4):
    hr_files = sorted(glob.glob(f'{DATA}/{split}/HR/*.png'))[:n]
    fig, ax = plt.subplots(2, len(hr_files), figsize=(3.2 * len(hr_files), 6.8))
    for i, f in enumerate(hr_files):
        stem = os.path.basename(f)[:-4]
        hr = imageio.imread(f)
        lr = imageio.imread(f'{DATA}/{split}/LR_bicubic/X3/{stem}x3.png')
        ax[0, i].imshow(lr); ax[0, i].set_title(f'LR {lr.shape[1]}x{lr.shape[0]}', fontsize=9)
        ax[1, i].imshow(hr); ax[1, i].set_title(f'HR {hr.shape[1]}x{hr.shape[0]}', fontsize=9)
        ax[1, i].set_xlabel(stem.rsplit('_y', 1)[0], fontsize=7)
        for a in (ax[0, i], ax[1, i]): a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f'{split}  (위: 입력 LR 10m / 아래: 정답 HR 3.33m)')
    plt.tight_layout(); plt.show()

preview('training')
preview('validation')

### 뭘 보면 되나

위가 흐린 입력, 아래가 선명한 정답이다. **위를 아래처럼 만드는 게 이 AI가 할 일이다.**

그리고 **연습문제(training)와 실전시험(validation)을 비교해서 보자.**

- 연습문제는 정답을 그대로 줄여서 만들었다. 그래서 위아래 색이 똑같다.
- 실전시험은 진짜 위성이 다른 날 찍은 사진이다. 위아래 색이 미묘하게 다르다.

이 색 차이가 나중에 점수를 눌러앉히는 범인이다.

## 3. 학습시키기

처음부터 가르치면 몇 시간이 걸린다. 그래서 **이미 어느 정도 배워둔 상태에서 이어서**
공부시킨다. 사람으로 치면 기초를 뗀 학생에게 심화 문제를 더 풀리는 것과 같다.

설정값은 아래 몇 개만 알면 된다.

| 값 | 뜻 |
|---|---|
| `EPOCHS='11'` | 문제집을 몇 번 반복해서 풀지. **11을 넣으면 실제로는 10번 돈다** (원본 코드의 버릇) |
| `TEST_EVERY='100'` | 한 번 돌 때 문제를 몇 개 풀지 |
| `LR='1e-4'` | 한 번에 얼마나 크게 고쳐잡을지. 크면 거칠고, 작으면 느리다 |
| `RESET='0'` | 연결이 끊겼을 때 이어서 하고 싶으면 이걸로 바꾼다 |

RTX 3090에서 40초, Colab T4에서는 3~5분쯤 걸린다.

In [ ]:
import time

env = dict(os.environ,
    GPU='0',
    DATA='COLAB',
    DIR_DATA=DATA,
    EPOCHS='11',            # 실제 10 epoch (EDSR off-by-one)
    DECAY='5-8',
    LR='1e-4',
    TEST_EVERY='100',       # 40패치짜리 소형셋 -> epoch당 1600샘플
    PRINT_EVERY='20',       # TEST_EVERY 보다 작아야 loss 로그가 남는다
    N_THREADS='2',          # Colab은 vCPU 2개
    SAVE='edsr_colab_x3',
    SAVE_RESULTS='0',
    RESET='1',              # 이어서 학습할 때는 '0'
    PRETRAIN=f'{MODEL}/checkpoints/edsr_ikonos_x3_best.pt',
)

t0 = time.time()
p = subprocess.run(['bash', f'{CODE}/run_train.sh'], env=env,
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout.splitlines():
    if 'Making a binary' in line or 'it/s]' in line: continue
    print(line)
print(f'\n소요 {time.time() - t0:.0f}초')

## 4. 얼마나 배웠나 그래프로 보기

두 개를 나란히 그린다.

- **왼쪽 — 연습문제 점수**: 낮을수록 잘 푸는 것 (틀린 정도라서 낮아야 좋다)
- **오른쪽 — 실전시험 점수**: 높을수록 잘 푸는 것 (PSNR, dB 단위)

In [ ]:
import numpy as np
import torch

EXP = f'{CODE}/experiment/edsr_colab_x3'
# EDSR 이 epoch 별 평균을 텐서로 저장해준다 (log.txt 를 정규식으로 긁는 것보다 안전)
loss = torch.load(f'{EXP}/loss_log.pt').flatten().numpy()
psnr = torch.load(f'{EXP}/psnr_log.pt').flatten().numpy()
ep = np.arange(1, len(loss) + 1)
best = int(np.argmax(psnr))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ep, loss, 'o-')
ax[0].set_title('학습 L1 loss — 합성 LR 도메인')
ax[0].set_xlabel('epoch'); ax[0].grid(alpha=.3)
ax[1].plot(ep, psnr, 'o-')
ax[1].plot(best + 1, psnr[best], 'r*', ms=15, label=f'best {psnr[best]:.3f} @ep{best+1}')
ax[1].set_title('검증 PSNR — 실제 S2 LR, 홀드아웃 10패치')
ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'학습 loss  {loss[0]:.2f} -> {loss[-1]:.2f}  (계속 내려간다)')
print(f'검증 PSNR  best {psnr[best]:.3f} dB @ epoch {best+1} / 마지막 {psnr[-1]:.3f} dB')

### 이상하지 않나?

**왼쪽은 계속 좋아지는데 오른쪽은 초반에 멈춘다.** 고장난 게 아니다.

연습문제는 정답을 줄여서 만든 거라 규칙이 일정하다. 그래서 계속 풀수록 잘 푼다.
그런데 실전시험은 진짜 위성이 다른 날 찍은 사진이라, 연습문제에서 익힌 규칙이
그대로 안 통한다.

깔끔한 문제집만 반복해서 푼 학생과 같다. 문제집 점수는 계속 오르는데 실전 점수는
어느 선에서 멈춘다. 남은 차이는 선명하게 만드는 실력 부족이 아니라 **위성이 다르고
찍은 날이 달라서 생긴 차이**다.

더 올리려면 연습문제를 실전과 비슷하게 만들어야 한다. 일부러 색을 틀어놓거나
노이즈를 섞는 식이다.

## 5. 진짜 사진에 써보기

이제 배운 걸 인천 위성사진 한 장에 적용한다. 2001×2001 픽셀짜리가 6003×6003으로 커진다.

사진이 커서 한 번에 넣으면 메모리가 터진다. 그래서 **작은 조각으로 잘라서 처리하고
다시 이어붙인다.** 조각 경계에 자국이 남지 않도록 살짝 겹쳐서 자른다.

결과는 위치 정보가 그대로 붙은 GeoTIFF로 저장되니까, QGIS 같은 지도 프로그램에서
원본 위에 바로 겹쳐볼 수 있다.

> `--chop` 옵션은 쓰지 마세요. 원본 코드에 버그가 있어서 켜면 에러가 납니다.

In [ ]:
OUTDIR = '/content/results'
weight = f'{CODE}/experiment/edsr_colab_x3/model/model_best.pt'

p = subprocess.run(['python', 'infer.py', '--weight', weight,
                    '--input', f'{DATA}/test', '--output', OUTDIR],
                   cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(p.stdout[-1500:])

In [ ]:
# 좌표가 보존됐는지 확인 — 실습에서 가장 자주 틀리는 부분이다
src_tif = glob.glob(f'{DATA}/test/*.tif')[0]
out_tif = glob.glob(f'{OUTDIR}/*_SRx3.tif')[0]

with rasterio.open(src_tif) as a, rasterio.open(out_tif) as b:
    print(f'입력  {a.width}x{a.height}  {a.res[0]:.4f} m  {a.crs}')
    print(f'출력  {b.width}x{b.height}  {b.res[0]:.4f} m  {b.crs}')
    same = np.allclose(np.array(a.bounds), np.array(b.bounds), atol=1e-6)
    print(f'\n지리 범위 일치: {same}')
    assert same, '좌표가 어긋났습니다'

## 6. 그냥 확대한 것과 비교하기

AI를 쓴 게 의미가 있었을까? 가장 단순한 확대 방법(bicubic, 주변 픽셀 평균내서 늘리기)과
비교해본다.

정답 사진이 없어서 점수는 못 낸다. 대신 두 가지로 판단한다.

- **선명도 수치(`lap_std`)** — 픽셀 사이 변화가 얼마나 급한지. 흐리면 낮고 또렷하면 높다.
- **눈으로 확인** — 건물 경계나 도로가 실제로 또렷해졌는지.

In [ ]:
import cv2

with rasterio.open(src_tif) as s:
    lr = np.ascontiguousarray(s.read([1, 2, 3]).transpose(1, 2, 0))
with rasterio.open(out_tif) as s:
    sr = np.ascontiguousarray(s.read([1, 2, 3]).transpose(1, 2, 0))
bic = cv2.resize(lr, (lr.shape[1] * 3, lr.shape[0] * 3), interpolation=cv2.INTER_CUBIC)

def lap_std(img):
    return float(cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), cv2.CV_64F).std())

print(f'Bicubic x3   lap_std {lap_std(bic):6.2f}   평균밝기 {bic.reshape(-1,3).mean(0).round(1)}')
print(f'EDSR    x3   lap_std {lap_std(sr):6.2f}   평균밝기 {sr.reshape(-1,3).mean(0).round(1)}')
print('\n※ lap_std는 같은 격자끼리만 비교할 것. 10m 원본 LR의 값은 픽셀 계단 때문에 크게 나온다.')

In [ ]:
# 텍스처가 많은 구역을 골라 확대 비교
gray = cv2.cvtColor(sr, cv2.COLOR_RGB2GRAY)
S = 300
best, bs = (0, 0), -1
for y in range(0, sr.shape[0] - S, S):
    for x in range(0, sr.shape[1] - S, S):
        v = gray[y:y+S, x:x+S].std()
        if v > bs: best, bs = (y, x), v
y, x = best

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
for a, img, t in zip(ax, [cv2.resize(lr[y//3:y//3+S//3, x//3:x//3+S//3], (S, S),
                                     interpolation=cv2.INTER_NEAREST), bic[y:y+S, x:x+S], sr[y:y+S, x:x+S]],
                     ['입력 LR (10 m, 최근접확대)', 'Bicubic ×3', 'EDSR ×3']):
    a.imshow(img); a.set_title(t); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 7. 결과 저장하기

Colab이 꺼지면 파일이 사라진다. 남기고 싶은 건 구글 드라이브로 옮긴다.

In [ ]:
if SAVE_TO_DRIVE:
    dst = '/content/drive/MyDrive/sr_colab/results'
    os.makedirs(dst, exist_ok=True)
    for f in glob.glob(f'{OUTDIR}/*'):
        shutil.copy(f, dst)
    print('저장:', os.listdir(dst))

---

## 자주 걸리는 함정

**1. `EPOCHS=10`을 넣으면 9번만 돈다.**
원본 코드가 그렇게 만들어져 있다. 10번 돌리고 싶으면 11을 넣자. `EPOCHS=1`은 아예
아무것도 배우지 않는다.

**2. `--chop` 옵션을 켜면 에러가 난다.**
원본 코드의 버그다. 큰 사진은 이 노트북처럼 조각내서 처리하면 된다.

**3. 자료를 바꿔 넣었는데 예전 것이 계속 나온다.**
`dataset/bin/` 폴더를 지우세요. 속도를 위해 사진을 미리 변환해 저장해두는데, 파일 이름이
같으면 예전 것을 그대로 쓴다.

**4. 학습이 이상하게 느리다.**
구글 드라이브에서 직접 학습시키면 안 된다. 드라이브는 인터넷 너머에 있어서 작은 파일을
많이 읽으면 아주 느려진다. 이 노트북처럼 Colab 안으로 복사한 뒤 학습해야 한다.

**5. 연결이 끊겨서 처음부터 다시 해야 한다.**
학습 폴더를 드라이브에 연결해뒀다면, `RESET`을 `'0'`으로 바꾸고 다시 실행하면 끊긴
지점부터 이어서 한다.

**6. 실전시험 점수가 낮은데 실패한 건가요?**
아니다. 실전시험은 진짜 위성이 찍은 다른 사진이라 원래 점수가 낮게 나온다.
중요한 건 절대값이 아니라 **그냥 확대한 것보다 나은가**이다.